# Hybrid Power Dispatch for Telecom Base Stations: A Computational Study

**CSC8101 Object-Oriented Programming, Advent 2026.** Tumuhirwe Iden (S26M25/011), Uganda Christian University

---

### Abstract

Many telecom base stations have a weak grid connection or no grid at all. When the grid fails,
they use diesel generators. Diesel is expensive, generators need maintenance, and they produce
carbon dioxide (CO₂).

This notebook uses the `hybrid_power` package to study a site with four energy sources:
**solar photovoltaic (PV) panels, the electricity grid, a battery and a diesel generator**.
Every hour, the site must decide which sources to use. This decision is called **dispatch**.

We compare three dispatch strategies on the same random operating data:

* **baseline**: the traditional method. It uses the grid when the grid is on and the generator
  when it is off. It does not use solar or the battery.
* **solar-first**: it always uses sources in a fixed order: solar, then battery, then grid, then
  generator. It does not look at grid prices or at the risk of a power cut.
* **optimiser**: a set of rules that look at grid prices and keep battery energy ready for
  power cuts.

The study does more than one simulation run. It:

1. checks that the simulator follows the basic laws of energy;
2. shows where the savings come from;
3. tests how much the results change when the inputs change;
4. repeats the simulation with many random patterns of weather and power cuts;
5. switches off the optimiser's rules one by one to see which ones are useful;
6. tests whether results from a few sites are also true for a large group of sites.

### Research questions

The study answers five research questions, called **RQ1** to **RQ5**. Each results section says
which question it answers. The symbol § means "section".

| | Question | Section |
|---|---|---|
| **RQ1** | Does the optimiser reduce cost, diesel use and emissions compared with the baseline, and is the power supply still reliable? | §5 to §6 |
| **RQ2** | Which inputs have the biggest effect on the saving? | §7 |
| **RQ3** | Does the saving stay the same with different weather and power cuts? Is the optimiser really better than solar-first? | §8 |
| **RQ4** | Which rules of the optimiser are the most useful? | §9 |
| **RQ5** | Are results from a few sites also true for a large group of sites? | §10 |

> **How to run.** Install with `pip install -e ".[notebook]"`, then open the notebook with
> `jupyter lab notebooks/hybrid_power_study.ipynb`. To run a smaller and faster version (as the
> automatic tests do), set the environment variable `HP_NOTEBOOK_FAST=1`. It then takes less than
> a minute. The comments in the text describe the results of the full-size run.

## Notation and abbreviations

Each symbol is also explained where it is first used. This list puts them all in one place.

### Abbreviations

| Term | Meaning |
|---|---|
| **RQ** | Research question (RQ1 to RQ5, see above) |
| **PV** | Photovoltaic: solar panels that turn sunlight into electricity |
| **SoC** | State of charge: how full the battery is, from 0 (empty) to 1 (full) |
| **KPI** | Key performance indicator: a number that measures a result, such as total cost or diesel used |
| **LPSP** | Loss of power supply probability: the share of demand that was not supplied (0 = always supplied) |
| **TOU** | Time-of-use price: the grid price is higher in the evening (18:00 to 23:00) than at other times |
| **CO₂** | Carbon dioxide, measured in kg |
| **kW / kWh** | Kilowatt is power; kilowatt-hour is energy. 1 kW for one hour = 1 kWh |
| **kWp** | Kilowatt-peak: the maximum power of solar panels in full sunshine |
| **L** | Litres of diesel |
| **MC** | Monte Carlo: running a simulation many times with different random data to see the range of results |
| **CI** | Confidence interval: a range that contains the true average with a stated probability (here 95%) |
| **SD / SE** | Standard deviation: how spread out single values are. Standard error: how uncertain an average is (SD/√n) |
| **pp** | Percentage points: the simple difference between two percentages (from 60% to 56% is −4 pp) |
| **OLS** | Ordinary least squares: a common method for fitting a straight-line (regression) model |
| **R²** | The share of the variation that a regression model explains (from 0 to 1) |
| **capex** | Capital expenditure: the cost of buying equipment |
| **HOMER** | A well-known software tool for designing small power systems. We use its generator fuel formula |
| **MPC** | Model-predictive control: a controller that plans ahead using forecasts |

### Mathematical symbols (§2)

| Symbol | Meaning | Unit |
|---|---|---|
| $t$, $T$ | Number of the hourly step; total number of steps ($T = 24 \times$ days) | – |
| $\Delta t$ | Length of one step (1 hour). In other places, $\Delta$ means "change in" | h |
| $D_t$ | Energy that the site needs (the demand) in step $t$ | kWh |
| $U_t$ | Unmet energy: demand that no source could supply in step $t$ | kWh |
| $E^{X\to Y}_t$ | Energy that flows from $X$ to $Y$ in step $t$ | kWh |
| $\mathrm{PV}, B, G, D, L$ | Labels: PV = solar, $B$ = battery, $G$ = grid, $D$ = diesel generator, $L$ = load | – |
| $E^{G}_t$ | All energy taken from the grid, $E^{G\to L}_t + E^{G\to B}_t$ | kWh |
| $E^{\mathrm{ch}}_t$ | All energy put into the battery (from solar, grid and generator) | kWh |
| $s_t$ | Solar factor: solar output as a share of the maximum output | 0–1 |
| $P^{\mathrm{PV}}_{\mathrm{rated}}$ | Maximum power of the solar panels | kWp |
| $C$ | Battery size (capacity) | kWh |
| $\eta_c$, $\eta_d$ | Charging / discharging efficiency (the share of energy that is not lost) | 0–1 |
| $\mathrm{SoC}_t$ | Battery state of charge at the end of step $t$ | 0–1 |
| $\mathrm{SoC}_{\min}$, $\mathrm{SoC}_{\max}$ | Lowest / highest allowed state of charge | 0–1 |
| $F_t$ | Diesel used in step $t$ | L |
| $P_{\mathrm{rated}}$ | Maximum power of the generator | kW |
| $P_t$ | Generator output in step $t$ (0 when it is off) | kW |
| $a$, $b$ | Fuel formula values: fuel to keep the engine running, and extra fuel per kWh | L/kWh |
| $\rho_{\min}$ | Lowest allowed generator output, as a share of $P_{\mathrm{rated}}$ | 0–1 |
| $\pi_t$ | Grid price in step $t$ | currency/kWh |
| $p_f$ | Diesel price | currency/L |
| $c_m$ | Generator maintenance cost for each hour it runs | currency/h |
| $c_{\deg}$ | Battery wear cost for each kWh that goes into or out of the battery | currency/kWh |
| $\mathbb 1[\cdot]$ | Equals 1 if the condition in the brackets is true, and 0 if it is false | – |
| $J$ | Total operating cost for the whole period | currency |
| $\varepsilon_G$, $\varepsilon_D$ | CO₂ produced by grid electricity and by diesel | kg/kWh, kg/L |
| $k_d$ | Daily clearness index: how clear the sky is on day $d$ (1 = no clouds) | 0–1 |
| $\lambda$ | Average number of power cuts per day | 1/day |
| $\tau$ | Average length of a power cut | h |
| $A$ | Grid availability: the share of hours when the grid is on | 0–1 |
| $n$ | Number of data sets (§8) or of sites (§10) | – |

## 1. Set-up and reproducibility

All random numbers in this study start from a fixed value called a **seed**. This means you get
exactly the same numbers every time you run the notebook. The second cell below sets the size of
each experiment.

In [1]:
from __future__ import annotations

import math
import os
import platform
import sys
import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

# Plotly displays figures in Jupyter via nbformat: fail early with a clear fix.
try:
    import nbformat  # noqa: F401
except ImportError as exc:
    raise ImportError(
        'Plotly needs nbformat to display figures. Run: pip install -e ".[notebook]"'
    ) from exc

# Make the package importable even when it has not been pip-installed.
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import hybrid_power as hp
from hybrid_power import (
    DispatchStrategy,
    FleetConfig,
    GeneratorMode,
    OptimizerConfig,
    RuleBasedOptimizer,
    ScenarioConfig,
    SimulationResult,
    SimulationService,
    StrategyRegistry,
    SyntheticProfileGenerator,
)
from hybrid_power.domain.dispatch import StepRecord
from hybrid_power.fleet import SummaryStatistics, run_fleet_study
from hybrid_power.presentation.charts import ChartFactory
from hybrid_power.simulation.engine import SimulationObserver

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_colwidth = 120

In [2]:
# ---- Experiment parameters -------------------------------------------------
FAST = os.environ.get("HP_NOTEBOOK_FAST", "0") == "1"

MC_REPLICATES = 8 if FAST else 60  # Monte Carlo profile seeds (§8)
SWEEP_POINTS = 3 if FAST else 7  # grid resolution of the 2-D sweep (§7.2)
TORNADO_SWING = 0.25  # +/- relative perturbation in the tornado (§7.1)
FLEET_SITES = 10 if FAST else 50  # sites in the fleet study (§10)
FLEET_DAYS = 7 if FAST else 30  # horizon per fleet site (§10)
CONFIDENCE = 0.95
EXPORT_DIR = ROOT / "results" / "notebook"

print(f"Fast mode: {FAST}")
print(
    f"Python {platform.python_version()} | hybrid_power {hp.__version__} | numpy {np.__version__}"
)
print(f"pandas {pd.__version__} | plotly {plotly.__version__}")

Fast mode: False
Python 3.12.2 | hybrid_power 1.0.0 | numpy 2.5.4
pandas 3.0.6 | plotly 7.1.0


## 2. System model

The simulator works in steps of one hour, $t = 1,\dots,T$. Here $T$ is the total number of hours,
and each step lasts $\Delta t = 1\,\mathrm{h}$.

In each step, the **dispatch strategy** receives the current situation of the site in a
read-only `DispatchContext`: the demand, the available solar energy, whether the grid is on, the
grid price, the battery level and the generator status. The strategy answers with a
`DispatchPlan`. The `BaseStation` first checks the whole plan, and it carries out the plan only
if every part is valid. In this way, the equipment can never be left in an impossible state.

**Energy balance.** The site's demand $D_t$ is supplied by solar, the battery, the grid and the
generator. If they cannot supply all of it, the missing part is recorded as unmet energy $U_t$:

$$
E^{\mathrm{PV}\to L}_t + E^{B\to L}_t + E^{G\to L}_t + E^{D\to L}_t + U_t = D_t .
$$

Here $E^{X\to Y}_t$ is the energy (kWh) that flows from $X$ to $Y$ in step $t$. The letters mean:
PV = solar panels, $B$ = battery, $G$ = grid, $D$ = diesel generator and $L$ = load (the
equipment that uses the power).

The solar energy available in one step is $P^{\mathrm{PV}}_{\mathrm{rated}}\,s_t\,\Delta t$.
Here $P^{\mathrm{PV}}_{\mathrm{rated}}$ is the maximum power of the panels (kWp), and $s_t$ is the
solar factor. It is a number between 0 and 1: 0 at night and close to 1 at midday on a clear day.
Solar energy goes to the load, to the battery, or it is wasted. Wasted solar energy is called
**curtailment**. In the same way, generator energy goes to the load, to the battery, or to a
**dump load**, which burns off extra energy that nothing can use.

**Battery.** $C$ is the battery size (kWh). $\eta_c$ and $\eta_d$ are the charging and
discharging efficiencies. For example, 0.95 means that 5% of the energy is lost.
$E^{\mathrm{ch}}_t = E^{\mathrm{PV}\to B}_t + E^{G\to B}_t + E^{D\to B}_t$ is all the energy put
into the battery in step $t$. The state of charge (SoC) changes like this:

$$
\mathrm{SoC}_{t} = \mathrm{SoC}_{t-1} + \frac{\eta_c\,E^{\mathrm{ch}}_t - E^{B\to L}_t/\eta_d}{C},
\qquad \mathrm{SoC}_{\min}\le\mathrm{SoC}_t\le\mathrm{SoC}_{\max}.
$$

When the battery charges, only the share $\eta_c$ of the input is stored. When it gives
$E^{B\to L}_t$ to the load, the stored energy falls by the larger amount $E^{B\to L}_t/\eta_d$.
The limits $\mathrm{SoC}_{\min}$ and $\mathrm{SoC}_{\max}$ (0.2 and 0.95 here) stop the battery from
becoming too empty or too full. This protects the battery's life.

**Diesel generator.** We calculate fuel use with the standard straight-line formula from HOMER
(Lambert *et al.*, 2006):

$$
F_t = \left(a\,P_{\mathrm{rated}} + b\,P_t\right)\Delta t \quad [\mathrm{L}], \qquad
a = 0.08415\ \mathrm{L/kWh},\; b = 0.246\ \mathrm{L/kWh},
$$

Here $F_t$ is the diesel used in step $t$ (litres), $P_{\mathrm{rated}}$ is the maximum power of
the generator (kW), and $P_t$ is its actual output in that step (kW). The first part, with $a$,
is the fuel that the engine uses just to keep running, even with no load. Because of this part,
a generator with a small load wastes a lot of fuel. The second part, with $b$, is the extra fuel
for each kWh produced. When the generator is on, its output must not fall below a minimum
level: $P_t \ge \rho_{\min} P_{\mathrm{rated}}$, with $\rho_{\min} = 0.3$ here.

**Operating cost and emissions.** $\pi_t$ is the grid price in step $t$ (per kWh). $p_f$ is the
price of one litre of diesel. $c_m$ is the generator maintenance cost for each hour it runs.
$c_{\deg}$ is the battery wear cost for each kWh that goes into or out of the battery. (All the
energy that goes in and out is called the *throughput*.) The total operating cost $J$ and the
total emissions are:

$$
J = \sum_t \Big[\pi_t E^{G}_t + p_f F_t + c_m\,\mathbb 1[P_t>0]\,\Delta t
+ c_{\deg}\big(E^{\mathrm{ch}}_t + E^{B\to L}_t\big)\Big],
\qquad
\mathrm{CO_2} = \sum_t \big(\varepsilon_G E^{G}_t + \varepsilon_D F_t\big).
$$

Here $E^{G}_t = E^{G\to L}_t + E^{G\to B}_t$ is all the energy taken from the grid. The term
$\mathbb 1[P_t>0]$ is 1 when the generator is running and 0 when it is off, so maintenance is
paid only for running hours. $\varepsilon_G$ is the CO₂ from 1 kWh of grid electricity (kg), and
$\varepsilon_D$ is the CO₂ from 1 litre of diesel (2.68 kg here).

**Reliability and renewable share.** The loss of power supply probability (LPSP) is the share
of demand that was not supplied. The renewable fraction $f_{\mathrm{ren}}$ is the share of the
produced energy that came from solar:

$$
\mathrm{LPSP} = \frac{\sum_t U_t}{\sum_t D_t},
\qquad
f_{\mathrm{ren}} = \frac{\sum_t \big(E^{\mathrm{PV}\to L}_t + E^{\mathrm{PV}\to B}_t\big)}
{\sum_t \big(E^{\mathrm{PV}\to L}_t + E^{\mathrm{PV}\to B}_t + E^{G}_t + E^{D\to L}_t + E^{D\to B}_t\big)} .
$$

**Savings.** We always compare a strategy with the baseline on the same data. The saving is
given as a percentage:

$$
\text{saving}\,(\%) = 100 \times \frac{J_{\text{baseline}} - J_{\text{candidate}}}{J_{\text{baseline}}},
$$

Fuel savings and CO₂ savings are calculated in the same way.

**Random inputs** (`SyntheticProfileGenerator`):

* **Demand** follows the typical daily (24-hour) pattern of a telecom site: low at night and
  high in the evening. A small random variation (normal, or Gaussian, noise) is added.
* **Solar** follows a curve shaped like half of a sine wave between 06:00 and 18:00. Each day,
  this curve is multiplied by a clearness index $k_d$, where 1 means a day with no clouds. The
  value of $k_d$ is chosen at random from a Beta distribution. Its average and spread can be set.
* **The grid** is either on or off. We model it as a two-state **Markov chain**: the state in the
  next hour depends only on the state now. When the grid is on, it fails with probability
  $\lambda/24$ each hour, where $\lambda$ is the average number of power cuts per day. When it is
  off, it comes back with probability $1/\tau$ each hour, where $\tau$ is the average length of a
  power cut in hours. Over a long time, the share of hours when the grid is on is:

$$
A = \frac{1/\tau}{1/\tau + \lambda/24}.
$$

The lengths of power cuts follow a **geometric distribution** with an average of $\tau$ hours.
This is the step-by-step version of the exponential distribution.

## 3. Reference scenario

Our main example is a rural telecom site in Northern Uganda (`data/sample_scenario.json`). It has
8 kWp of solar panels, a 30 kWh battery, a 10 kW diesel generator and a grid with a time-of-use
(TOU) price. On average, the grid fails about 1.2 times per day.

The scenario is stored in a `ScenarioConfig` object. This object cannot be changed after it is
created, and it checks every value. So a wrong value is rejected at once, not in the middle of a
simulation.

In [3]:
scenario = ScenarioConfig.load(ROOT / "data" / "sample_scenario.json")


def config_table(config: ScenarioConfig) -> pd.DataFrame:
    # Flatten the nested scenario into a (section, parameter) -> value table.
    rows = [
        (section, key, value)
        for section, values in config.to_dict().items()
        if isinstance(values, dict)
        for key, value in values.items()
    ]
    return pd.DataFrame(rows, columns=["section", "parameter", "value"]).set_index(
        ["section", "parameter"]
    )


display(Markdown(f"**{scenario.name}** (currency: {scenario.currency})"))
config_table(scenario)

**Sample rural macro site (Northern Uganda)** (currency: USD)

value
section   parameter                                   
profile   days                                      30
          start_date                        2026-10-01
          base_load_kw                           2.200
          peak_load_kw                           4.000
          load_noise                             0.050
          solar_mean_clearness                   0.700
          solar_variability                      0.300
          outages_per_day                        1.200
          mean_outage_hours                      3.000
          peak_tariff                            0.240
          offpeak_tariff                         0.120
          peak_start_hour                           18
          peak_end_hour                             23
          seed                                      42
solar     rated_kw                               8.000
          om_cost_per_kwh                        0.000
grid      max_import_kw                         15.000
          emission_factor_kg_per_kwh             0.100
battery   capacity_kwh                          30.000
          initial_soc                            0.800
          min_soc                                0.200
          max_soc                                0.950
          charge_efficiency                      0.950
          discharge_efficiency                   0.950
          max_charge_kw                         10.000
          max_discharge_kw                      10.000
          degradation_cost_per_kwh               0.020
generator rated_kw                              10.000
          tank_capacity_l                      150.000
          initial_fuel_l                       150.000
          fuel_price_per_l                       1.450
          min_load_ratio                         0.300
          fuel_curve_slope                       0.246
          fuel_curve_intercept                   0.084
          maintenance_cost_per_hour              0.400
          co2_kg_per_litre                       2.680
          refuel_interval_hours                    168
optimizer reserve_soc                            0.500
          peak_tariff_threshold                  0.200
          grid_charge_tariff_threshold           0.130
          grid_charge_target_soc                 0.700
          generator_mode                cycle_charging
          cycle_charge_load_ratio                0.800
          generator_stop_soc                     0.600

## 4. Exploratory analysis of the operating profile

Before we run the simulation, we check that the generated data looks correct. The data has one
row for each hour:

* `demand_kw` is the site demand ($D_t$);
* `solar_factor` is $s_t$;
* `grid_available` shows whether the grid is on;
* `tariff_per_kwh` is the grid price $\pi_t$.

In [4]:
profile = SyntheticProfileGenerator(scenario.profile).get_profile()
charts = ChartFactory(currency=scenario.currency)
charts.profile_overview(profile, scenario.solar.rated_kw).show()

profile_frame = profile.to_frame()
profile_frame.describe().T

,count,mean,min,25%,50%,75%,max,std
timestamp,720,2026-10-15 23:30:00,2026-10-01 00:00:00,2026-10-08 11:45:00,2026-10-15 23:30:00,2026-10-23 11:15:00,2026-10-30 23:00:00,NaN
demand_kw,720.000,3.306,2.125,2.768,3.476,3.755,4.583,0.564
solar_factor,720.000,0.213,0.000,0.000,0.000,0.440,0.980,0.271
tariff_per_kwh,720.000,0.145,0.120,0.120,0.120,0.120,0.240,0.049


### 4.1 Checking the power-cut model

From the Markov chain we can calculate two values in advance: the share of time the grid is on
($A$), and the average length of a power cut ($\tau$). Below we compare these expected values with
the values in the generated data. To reduce random variation, we combine 20 different seeds, with
90 days of data each. The column `rel. error` is (measured ÷ expected) − 1.

In [5]:
def outage_runs(available: np.ndarray) -> np.ndarray:
    # Lengths (in steps) of each contiguous run of grid outage.
    down = np.r_[False, ~available.astype(bool), False].astype(int)
    edges = np.flatnonzero(np.diff(down))
    return edges[1::2] - edges[::2]


cfg = scenario.profile
availability, runs = [], []
for seed in range(20):
    sample = SyntheticProfileGenerator(replace(cfg, seed=seed, days=90)).get_profile()
    availability.append(sample.grid_availability_ratio)
    runs.extend(outage_runs(sample.grid_available))
runs_arr = np.asarray(runs)

fail, repair = cfg.outages_per_day / 24, 1 / cfg.mean_outage_hours
validation = pd.DataFrame(
    {
        "theory": [
            repair / (repair + fail),
            cfg.mean_outage_hours,
            cfg.outages_per_day * repair / (repair + fail),
        ],
        "empirical": [
            np.mean(availability),
            runs_arr.mean(),
            len(runs_arr) / (20 * 90),
        ],
    },
    index=["grid availability A", "mean outage length τ (h)", "outages started per day λA"],
)
validation["rel. error"] = validation["empirical"] / validation["theory"] - 1
validation

,theory,empirical,rel. error
grid availability A,0.870,0.869,-0.001
mean outage length τ (h),3.000,2.997,-0.001
outages started per day λA,1.043,1.051,0.007


All three measured values are within about 1% of the expected values. Note one detail: power
cuts start at a rate of $\lambda A$, not $\lambda$. This is because the grid can only fail when it
is on. If we compared with $\lambda$, we would wrongly see a 13% difference. The chart below also
shows that the lengths of power cuts follow the expected geometric shape.

In [6]:
lengths = np.arange(1, runs_arr.max() + 1)
geometric = repair * (1 - repair) ** (lengths - 1)
fig = go.Figure()
fig.add_bar(
    x=lengths,
    y=np.bincount(runs_arr, minlength=lengths.max() + 1)[1:] / len(runs_arr),
    name="empirical",
)
fig.add_scatter(
    x=lengths, y=geometric, mode="lines+markers", name=f"Geometric(p=1/{cfg.mean_outage_hours:g})"
)
fig.update_layout(
    title="Outage duration distribution",
    xaxis_title="outage length (h)",
    yaxis_title="probability",
    template="plotly_white",
    height=380,
)
fig.show()

## 5. Comparing the three strategies (RQ1)

`SimulationService` is a simple front end (a *facade*). It connects the scenario, the list of
strategies, the simulation engine and the analysis code. We also add a small **observer**. An
observer is an object that is told about every step of the simulation. Our observer measures how
long each strategy takes to run. It also counts **generator switching events**, which means each
time the generator turns on or off.

In [7]:
class RunStatistics(SimulationObserver):
    # Observer collecting per-strategy timing and generator switching events.

    def __init__(self) -> None:
        self.rows: list[dict[str, object]] = []

    def on_start(self, strategy: DispatchStrategy, total_steps: int) -> None:
        self._name, self._steps, self._switches = strategy.name, total_steps, 0
        self._was_on, self._t0 = False, time.perf_counter()

    def on_step(self, record: StepRecord) -> None:
        self._switches += record.generator_on != self._was_on
        self._was_on = record.generator_on

    def on_complete(self, result: SimulationResult) -> None:
        elapsed = time.perf_counter() - self._t0
        self.rows.append(
            {
                "strategy": self._name,
                "steps": self._steps,
                "generator switching events": self._switches,
                "runtime (ms)": 1e3 * elapsed,
                "steps / s": self._steps / elapsed,
            }
        )


stats = RunStatistics()
service = SimulationService()
outcome = service.run(scenario, observers=[stats])
results, report = outcome.results, outcome.report
pd.DataFrame(stats.rows).set_index("strategy")

,steps,generator switching events,runtime (ms),steps / s
strategy,,,,
baseline,720,62,17.446,"41,271.192"
solar_first,720,40,20.305,"35,459.973"
optimizer,720,12,20.528,"35,073.475"


**KPI columns.**

* `total_cost` is the total cost $J$.
* `cost_per_kwh_served` is $J$ divided by the energy that was actually supplied.
* `fuel_used_l` is the total diesel used, $\sum_t F_t$.
* `generator_hours` is the number of hours the generator ran.
* `generator_starts` is the number of times the generator was switched on. More starts mean more
  engine wear.
* `emissions_kg` is the total CO₂.
* `renewable_fraction` is $f_{\mathrm{ren}}$.
* `solar_curtailed_kwh` is solar energy that was available but not used.
* `battery_throughput_kwh` is the energy that went into and out of the battery.
* `unmet_energy_kwh` is $\sum_t U_t$, and `lpsp` is the LPSP.

In [8]:
KPI_COLUMNS = [
    "total_cost",
    "cost_per_kwh_served",
    "fuel_used_l",
    "generator_hours",
    "generator_starts",
    "emissions_kg",
    "renewable_fraction",
    "solar_curtailed_kwh",
    "battery_throughput_kwh",
    "unmet_energy_kwh",
    "lpsp",
]
kpis = report.to_frame()[KPI_COLUMNS]
kpis

,total_cost,cost_per_kwh_served,fuel_used_l,generator_hours,generator_starts,emissions_kg,renewable_fraction,solar_curtailed_kwh,battery_throughput_kwh,unmet_energy_kwh,lpsp
strategy_name,,,,,,,,,,,
baseline,549.890,0.231,136.173,82.000,31,577.195,0.000,"1,225.066",0.000,0.000,0.000
solar_first,322.366,0.135,83.044,52.000,20,324.834,0.513,0.000,517.759,0.000,0.000
optimizer,239.720,0.101,30.905,11.000,6,201.324,0.474,76.603,855.049,0.000,0.000


In [9]:
rows = []
for name, cmp in report.comparisons().items():
    rows.append(
        {
            "strategy": name,
            "cost saving (%)": cmp.cost_savings_pct,
            "fuel saving (%)": cmp.fuel_savings_pct,
            "CO2 reduction (%)": cmp.emissions_reduction_pct,
            "generator hours avoided": cmp.generator_hours_reduction,
            "change in unmet energy (kWh)": cmp.unmet_energy_change_kwh,
            "improvement": cmp.is_improvement,
        }
    )
savings = pd.DataFrame(rows).set_index("strategy")
savings

,cost saving (%),fuel saving (%),CO2 reduction (%),generator hours avoided,change in unmet energy (kWh),improvement
strategy,,,,,,
solar_first,41.376,39.016,43.722,30.000,0.000,True
optimizer,56.406,77.305,65.120,71.000,0.000,True


### 5.1 Checking the simulator

Results are only useful if the simulator is correct. So, before we look at the results, we check
these rules for every hour and every strategy:

1. the energy balance of §2 is correct;
2. all solar energy goes somewhere (to the load, to the battery or to curtailment);
3. the battery level follows the battery equation of §2 and stays between
   $\mathrm{SoC}_{\min}$ and $\mathrm{SoC}_{\max}$;
4. no energy is taken from the grid during a power cut;
5. the costs of all the hours add up to the total cost.

A **residual** is the left side of an equation minus the right side. If the equation is correct,
the residual is 0, apart from tiny rounding errors in the computer. The table shows the largest
residual over all hours. The `assert` lines stop the notebook if any residual is larger than
$10^{-6}$.

In [10]:
bat = scenario.battery


def invariant_residuals(name: str, frame: pd.DataFrame) -> pd.Series:
    charge = frame.solar_to_battery + frame.grid_to_battery + frame.generator_to_battery
    soc_prev = np.r_[bat.initial_soc, frame.soc.to_numpy()[:-1]]
    soc_model = (
        soc_prev
        + (bat.charge_efficiency * charge - frame.battery_to_load / bat.discharge_efficiency)
        / bat.capacity_kwh
    )
    load_balance = (
        frame.solar_to_load
        + frame.battery_to_load
        + frame.grid_to_load
        + frame.generator_to_load
        + frame.unmet_kwh
        - frame.demand_kwh
    )
    pv_balance = (
        frame.solar_to_load
        + frame.solar_to_battery
        + frame.solar_curtailed
        - frame.solar_available_kwh
    )
    return pd.Series(
        {
            "max |load balance| (kWh)": load_balance.abs().max(),
            "max |PV balance| (kWh)": pv_balance.abs().max(),
            "max |SoC model error|": np.abs(soc_model - frame.soc).max(),
            "SoC below min (steps)": int((frame.soc < bat.min_soc - 1e-9).sum()),
            "SoC above max (steps)": int((frame.soc > bat.max_soc + 1e-9).sum()),
            "grid import in outage (kWh)": (
                (frame.grid_to_load + frame.grid_to_battery)[~frame.grid_available.astype(bool)]
            ).sum(),
            "|sum step cost - KPI|": abs(frame.total_cost.sum() - report.metrics[name].total_cost),
        }
    )


checks = pd.DataFrame({name: invariant_residuals(name, r.frame) for name, r in results.items()})
display(checks)

TOL = 1e-6
assert (checks.loc[checks.index.str.startswith("max")] < TOL).all().all(), (
    "energy/SoC invariant violated"
)
assert (checks.loc[["SoC below min (steps)", "SoC above max (steps)"]] == 0).all().all()
assert (checks.loc["grid import in outage (kWh)"] < TOL).all()
assert (checks.loc["|sum step cost - KPI|"] < TOL).all()
print("All invariants hold for every strategy and every step.")

,baseline,solar_first,optimizer
max |load balance| (kWh),0.000,0.000,0.000
max |PV balance| (kWh),0.000,0.000,0.000
max |SoC model error|,0.000,0.000,0.000
SoC below min (steps),0.000,0.000,0.000
SoC above max (steps),0.000,0.000,0.000
grid import in outage (kWh),0.000,0.000,0.000
|sum step cost - KPI|,0.000,0.000,0.000


All invariants hold for every strategy and every step.


### 5.2 Cost, energy and environmental results

In [11]:
metrics = report.ranking()
charts.cost_breakdown(metrics).show()
charts.energy_mix(metrics).show()
charts.environmental_impact(metrics).show()
charts.cumulative_cost(results).show()

### 5.3 Where does the saving come from?

Here we split the cost difference between the baseline and each strategy into three parts: grid,
generator and battery. A positive bar is money saved. A negative bar is a new cost, for example
battery wear, that the strategy accepts in order to save money in another part.

In [12]:
components = ["grid_cost", "generator_cost", "battery_cost"]
base = report.metrics[report.baseline_name]
attribution = pd.DataFrame(
    {
        name: {c.replace("_cost", ""): getattr(base, c) - getattr(m, c) for c in components}
        for name, m in report.metrics.items()
        if name != report.baseline_name
    }
).T
attribution["net saving"] = attribution.sum(axis=1)
display(attribution)

fig = px.bar(
    attribution.drop(columns="net saving")
    .reset_index(names="strategy")
    .melt(id_vars="strategy", var_name="component", value_name="saving"),
    x="strategy",
    y="saving",
    color="component",
    barmode="relative",
    title=f"Attribution of cost savings vs baseline ({scenario.currency})",
    template="plotly_white",
    height=400,
)
fig.add_scatter(
    x=attribution.index,
    y=attribution["net saving"],
    mode="markers",
    marker={"symbol": "diamond", "size": 12, "color": "black"},
    name="net",
)
fig.show()

,grid,generator,battery,net saving
solar_first,148.843,89.037,-10.355,227.525
optimizer,146.232,181.039,-17.101,310.170


### 5.4 How the optimiser works

The charts show one week. They show *how* the optimiser saves money:

* in the evening, when the grid price is high, it uses the battery, but it keeps some energy for
  power cuts;
* at night, when the grid price is low, it charges the battery from the grid;
* in cycle-charging mode, it runs the generator at an efficient level and stores the extra energy
  in the battery.

In [13]:
start = profile.timestamps[0].to_pydatetime()
week = {
    k: r.between(start, start + pd.Timedelta(days=7).to_pytimedelta()) for k, r in results.items()
}
charts.power_flow(week["optimizer"]).show()
charts.battery_soc(week).show()

In [14]:
charts.hourly_heatmap(results["optimizer"], "Operating cost").show()
charts.savings_calendar(results["baseline"], results["optimizer"]).show()

**Explaining decisions.** For every hour, the simulator records the reasons for its decision.
The table below shows the optimiser's decisions in the first hours without grid power.

In [15]:
log = results["optimizer"].frame
log.loc[~log.grid_available.astype(bool), ["soc", "generator_on", "reasons"]].head(10)

,soc,generator_on,reasons
timestamp,,,
2026-10-01 16:00:00,0.950,False,Solar supplies 3.74 kWh (zero fuel cost and emissions)
2026-10-01 17:00:00,0.893,False,"Solar supplies 1.98 kWh (zero fuel cost and emissions) | Grid outage: battery carries the load (1.62 kWh), no start"
2026-10-01 18:00:00,0.750,False,"Grid outage: battery carries the load (4.08 kWh), no start"
2026-10-03 11:00:00,0.717,False,Solar supplies 3.82 kWh (zero fuel cost and emissions) | Surplus solar charges battery with 0.08 kWh
2026-10-06 00:00:00,0.609,False,"Grid outage: battery carries the load (2.60 kWh), no start"
2026-10-06 08:00:00,0.713,False,Solar supplies 3.24 kWh (zero fuel cost and emissions) | Surplus solar charges battery with 0.42 kWh
2026-10-06 09:00:00,0.765,False,Solar supplies 3.33 kWh (zero fuel cost and emissions) | Surplus solar charges battery with 1.63 kWh
2026-10-06 10:00:00,0.844,False,Solar supplies 3.50 kWh (zero fuel cost and emissions) | Surplus solar charges battery with 2.49 kWh
2026-10-06 22:00:00,0.390,False,"Grid outage: battery carries the load (3.12 kWh), no start"


## 6. Answer to RQ1

At the reference site, the optimiser has the lowest cost, diesel use, emissions and generator
hours of the three strategies. Like the baseline, it supplies all the demand (LPSP = 0).

The breakdown in §5.3 shows that the two strategies save money in different ways. Solar-first and
the optimiser both use about the same amount of grid energy. The optimiser saves more mainly
because it uses the generator much less. During most power cuts it supplies the site from the
battery, so it starts the generator far less often. In exchange, its battery wears more and it
wastes some solar energy. For this reason, its renewable fraction is a little *lower* than
solar-first's, even though it is cheaper. If we judged the strategies only by renewable share,
we would choose the wrong one.

However, this is only one 30-day run with one pattern of weather and power cuts. Sections 7 and 8
test whether the conclusion is still true in other conditions.

## 7. Sensitivity analysis (RQ2)

### 7.1 One input at a time (tornado chart)

We change one input at a time by 25% (`TORNADO_SWING`), first down and then up from its reference
value. All other inputs stay the same. We also keep the same random weather and power cuts (the
same seed). This method is called **common random numbers**. Because of it, any change in the
result comes from the input, not from different random data.

We record the optimiser's cost saving compared with the baseline. In the table:

* `low` and `high` are the savings at −25% and +25%;
* `range (pp)` is the difference between them, in percentage points.

The chart is called a **tornado chart**: when the bars are sorted by length, the shape looks like
a tornado. The dashed line shows the saving at the reference values.

In [16]:
def with_param(config: ScenarioConfig, section: str, field: str, value: float) -> ScenarioConfig:
    return replace(config, **{section: replace(getattr(config, section), **{field: value})})


def saving_pct(config: ScenarioConfig, candidate: str = "optimizer") -> float:
    rep = service.run(config, strategies=["baseline", candidate]).report
    return float(rep.comparison(candidate).cost_savings_pct)


PARAMETERS = {
    "diesel price": ("generator", "fuel_price_per_l"),
    "battery capacity": ("battery", "capacity_kwh"),
    "PV rating": ("solar", "rated_kw"),
    "outages per day": ("profile", "outages_per_day"),
    "mean outage length": ("profile", "mean_outage_hours"),
    "peak tariff": ("profile", "peak_tariff"),
    "site peak load": ("profile", "peak_load_kw"),
}

reference_saving = saving_pct(scenario)
rows = []
for label, (section, field) in PARAMETERS.items():
    ref = getattr(getattr(scenario, section), field)
    low = saving_pct(with_param(scenario, section, field, ref * (1 - TORNADO_SWING)))
    high = saving_pct(with_param(scenario, section, field, ref * (1 + TORNADO_SWING)))
    rows.append({"parameter": label, "reference": ref, "low": low, "high": high})

tornado = pd.DataFrame(rows).set_index("parameter")
tornado["range (pp)"] = (tornado.high - tornado.low).abs()
tornado = tornado.sort_values("range (pp)")
display(tornado)

fig = go.Figure()
fig.add_bar(
    y=tornado.index,
    x=tornado.low - reference_saving,
    base=reference_saving,
    orientation="h",
    name=f"-{TORNADO_SWING:.0%}",
)
fig.add_bar(
    y=tornado.index,
    x=tornado.high - reference_saving,
    base=reference_saving,
    orientation="h",
    name=f"+{TORNADO_SWING:.0%}",
)
fig.add_vline(x=reference_saving, line_dash="dash")
fig.update_layout(
    barmode="overlay",
    template="plotly_white",
    height=420,
    title="Tornado: optimiser cost saving vs baseline (%)",
    xaxis_title="cost saving (%)",
)
fig.show()

,reference,low,high,range (pp)
parameter,,,,
mean outage length,3.000,56.750,57.189,0.439
diesel price,1.450,54.345,58.127,3.783
peak tariff,0.240,60.756,56.157,4.599
battery capacity,30.000,54.302,58.969,4.667
outages per day,1.200,53.605,58.746,5.142
PV rating,8.000,49.463,59.889,10.425
site peak load,4.000,64.725,49.017,15.709


### 7.2 Battery size and solar size together

The battery size and the solar size affect each other. For example, a larger battery is not
useful if there is no extra solar energy to fill it. So we change both sizes together and show
the saving and the diesel use of the optimiser.

In [17]:
capacities = np.linspace(10, 60, SWEEP_POINTS)
pv_ratings = np.linspace(2, 14, SWEEP_POINTS)
grid_rows = []
for cap in capacities:
    for pv in pv_ratings:
        cfg_ = with_param(
            with_param(scenario, "battery", "capacity_kwh", float(cap)),
            "solar",
            "rated_kw",
            float(pv),
        )
        rep = service.run(cfg_, strategies=["baseline", "optimizer"]).report
        grid_rows.append(
            {
                "battery_kwh": cap,
                "pv_kw": pv,
                "saving_pct": rep.comparison("optimizer").cost_savings_pct,
                "optimizer_cost": rep.metrics["optimizer"].total_cost,
                "fuel_l": rep.metrics["optimizer"].fuel_used_l,
            }
        )
sweep = pd.DataFrame(grid_rows)

fig = px.imshow(
    sweep.pivot(index="battery_kwh", columns="pv_kw", values="saving_pct").sort_index(
        ascending=False
    ),
    text_auto=".0f",
    color_continuous_scale="Viridis",
    aspect="auto",
    labels={"x": "PV rating (kWp)", "y": "battery capacity (kWh)", "color": "saving %"},
    title="Optimiser cost saving vs baseline (%)",
)
fig.update_layout(height=450)
fig.show()

fig = px.line(
    sweep,
    x="pv_kw",
    y="fuel_l",
    color="battery_kwh",
    markers=True,
    title="Optimiser diesel use vs PV rating, by battery size",
    labels={"pv_kw": "PV rating (kWp)", "fuel_l": "diesel (L)", "battery_kwh": "battery (kWh)"},
    template="plotly_white",
    height=420,
)
fig.show()

**Answer to RQ2.** The saving depends mostly on **how much solar there is compared with the
demand**. The site's peak demand and the solar size have by far the longest bars in the tornado
chart. A higher demand makes the saving smaller. The power-cut inputs and the diesel price matter
much less. This is because the optimiser already supplies most power cuts from the battery, so
there is little diesel left to save.

The second experiment shows how the two sizes work together. With a fixed battery, more solar
stops helping at about 10 kWp, because the battery is already full and the extra solar energy is
wasted. A larger battery keeps reducing diesel use, until diesel use reaches zero at about 60 kWh.

Note that this experiment is only a first check. It does not include the cost of buying or
replacing equipment (see §11).

## 8. Testing with random conditions (RQ3)

Now we treat weather, demand and power cuts as random. We create `MC_REPLICATES` different data
sets, one for each seed, and we run all three strategies on the *same* data set each time.
Because the strategies see the same conditions, we can compare them in **pairs**. This removes
the effect of good or bad weather from the comparison.

For each value we report the average and a 95% **bootstrap** confidence interval (Efron &
Tibshirani, 1993), calculated by `SummaryStatistics`. The bootstrap works like this. It picks
$n$ values at random from our $n$ results (the same value can be picked more than once) and
calculates their average. It repeats this 2,000 times. `ci_low` and `ci_high` are the 2.5% and
97.5% points of these 2,000 averages. This method does not need to assume that the results follow
a normal distribution.

To compare the optimiser with solar-first, we use a **sign test** on the paired cost differences.
If both strategies were equally good, each one would win a data set with a probability of ½. The
**p-value** is the chance of a result at least as one-sided as ours if this were true. A very
small p-value means that the difference is very unlikely to be caused by chance.

In [18]:
mc_rows = []
for seed in range(MC_REPLICATES):
    cfg_ = with_param(scenario, "profile", "seed", seed)
    rep = service.run(cfg_).report
    row = {"seed": seed}
    for name, m in rep.metrics.items():
        row[f"{name}_cost"] = m.total_cost
        row[f"{name}_fuel"] = m.fuel_used_l
        row[f"{name}_lpsp"] = m.loss_of_power_supply_probability
    for name, cmp in rep.comparisons().items():
        row[f"{name}_saving_pct"] = cmp.cost_savings_pct
    mc_rows.append(row)
mc = pd.DataFrame(mc_rows).set_index("seed")
mc.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
baseline_cost,576.496,46.403,474.135,675.290
baseline_fuel,158.384,33.000,82.423,230.332
baseline_lpsp,0.000,0.000,0.000,0.000
solar_first_cost,316.746,31.538,250.645,392.958
solar_first_fuel,87.084,20.775,41.067,136.713
solar_first_lpsp,0.000,0.000,0.000,0.000
optimizer_cost,218.554,14.889,192.688,263.303
optimizer_fuel,19.386,10.735,0.000,50.571
optimizer_lpsp,0.000,0.000,0.000,0.000
solar_first_saving_pct,45.014,3.930,35.706,52.906


In [19]:
def summarise(values: pd.Series) -> dict[str, float]:
    s = SummaryStatistics.from_values(values.to_numpy(), confidence=CONFIDENCE, seed=0)
    return {"mean": s.mean, "std": s.std, "ci_low": s.ci_low, "ci_high": s.ci_high, "n": s.count}


uq = pd.DataFrame(
    {
        "optimizer saving vs baseline (%)": summarise(mc.optimizer_saving_pct),
        "solar_first saving vs baseline (%)": summarise(mc.solar_first_saving_pct),
        "optimizer fuel (L)": summarise(mc.optimizer_fuel),
        "baseline fuel (L)": summarise(mc.baseline_fuel),
        "optimizer LPSP": summarise(mc.optimizer_lpsp),
    }
).T
uq

,mean,std,ci_low,ci_high,n
optimizer saving vs baseline (%),61.922,3.219,61.123,62.701,60.000
solar_first saving vs baseline (%),45.014,3.930,43.999,46.011,60.000
optimizer fuel (L),19.386,10.735,16.670,22.103,60.000
baseline fuel (L),158.384,33.000,150.540,166.669,60.000
optimizer LPSP,0.000,0.000,0.000,0.000,60.000


In [20]:
def sign_test(diff: np.ndarray) -> tuple[int, int, float]:
    # Exact two-sided sign test on non-zero paired differences.
    nonzero = diff[np.abs(diff) > 1e-9]
    n, wins = len(nonzero), int((nonzero > 0).sum())
    k = min(wins, n - wins)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2**n) if n else 1.0
    return wins, n, p


paired = (mc.solar_first_cost - mc.optimizer_cost).to_numpy()  # >0: optimizer cheaper
wins, n, p_value = sign_test(paired)
diff_stats = SummaryStatistics.from_values(paired, confidence=CONFIDENCE, seed=1)
display(
    Markdown(
        f"The optimiser was cheaper than solar-first on **{wins} of {n}** data sets. "
        f"On average it saved **{diff_stats.mean:.2f} {scenario.currency}** per data set "
        f"({CONFIDENCE:.0%} CI {diff_stats.ci_low:.2f} to {diff_stats.ci_high:.2f}). "
        f"Sign test p-value: **{p_value:.2g}**."
    )
)

fig = go.Figure()
for name in ["optimizer", "solar_first"]:
    fig.add_histogram(x=mc[f"{name}_saving_pct"], name=name, opacity=0.65, nbinsx=20)
fig.update_layout(
    barmode="overlay",
    template="plotly_white",
    height=400,
    title="Monte Carlo distribution of cost saving vs baseline",
    xaxis_title="cost saving (%)",
    yaxis_title="number of data sets",
)
fig.show()

The optimiser was cheaper than solar-first on **60 of 60** data sets. On average it saved **98.19 USD** per data set (95% CI 92.12 to 104.75). Sign test p-value: **1.7e-18**.

The next chart shows the average saving after $k$ runs. The shaded area is $\pm 1.96$
standard errors (SE $= \mathrm{SD}/\sqrt{k}$). The number 1.96 gives a 95% interval for a normal
distribution. When the shaded area becomes narrow and flat, more runs would not change the
result.

In [21]:
# Convergence of the Monte Carlo estimate: running mean with a +/- 1.96 SE band.
s = mc.optimizer_saving_pct.to_numpy()
k = np.arange(1, len(s) + 1)
running_mean = np.cumsum(s) / k
running_se = np.array([s[:i].std(ddof=1) / np.sqrt(i) if i > 1 else np.nan for i in k])
fig = go.Figure(
    [
        go.Scatter(x=k, y=running_mean + 1.96 * running_se, line={"width": 0}, showlegend=False),
        go.Scatter(
            x=k,
            y=running_mean - 1.96 * running_se,
            fill="tonexty",
            line={"width": 0},
            name="±1.96 SE",
            fillcolor="rgba(99,110,250,0.2)",
        ),
        go.Scatter(x=k, y=running_mean, name="running mean", line={"color": "#636efa"}),
    ]
)
fig.update_layout(
    template="plotly_white",
    height=360,
    xaxis_title="replicates",
    yaxis_title="mean saving (%)",
    title="Monte Carlo convergence",
)
fig.show()

**Answer to RQ3.** In all the random conditions, the optimiser still saves money. The
confidence interval is only about ±1 percentage point wide, so the result of RQ1 was not just
luck. In fact, the reference data set is a little *worse* for the optimiser than the average.
The optimiser's cost also changes the least of the three strategies. This is important for an
operator who must plan a budget for the most expensive month.

In the paired comparison, the optimiser beats solar-first on every data set. The sign test shows
that this is very clearly not chance. In fast mode there are only a few runs, so the p-value is
less reliable.

## 9. Switching off rules one at a time (RQ4)

The optimiser has several rules. To find out how much each rule helps, we switch off one rule at
a time and measure the change in the saving. This is called an **ablation study**.

| Variant | Rule that is switched off | How |
|---|---|---|
| `no outage reserve` | Keeping battery energy for the next power cut | `reserve_soc = 0` |
| `no grid charging` | Charging the battery with cheap grid power at night | `grid_charge_tariff_threshold = 0` |
| `no peak shaving` | Using the battery when the evening grid price is high | `peak_tariff_threshold = 10` (never reached) |
| `load following` | **Cycle charging**: running the generator at 80% of its maximum power and storing the extra energy | the generator produces only what the load needs |

This also tests the **open/closed principle** of the design: code should be *open* to new
features but *closed* to changes. We create the variants without changing the package. We write
a subclass of `RuleBasedOptimizer` and register it in our own `StrategyRegistry`. The engine and
the analysis code stay the same.

In the summary table, `Δ saving vs full (pp)` is the variant's average saving minus the full
optimiser's average saving, in percentage points. A negative value means that switching off the
rule loses money.

In [22]:
class NamedOptimizer(RuleBasedOptimizer):
    # A RuleBasedOptimizer variant reported under its own name.

    def __init__(self, name: str, config: OptimizerConfig) -> None:
        super().__init__(config)
        self._variant = name

    @property
    def name(self) -> str:
        return self._variant


base_rules = scenario.optimizer
VARIANTS = {
    "optimizer": base_rules,
    "no outage reserve": replace(base_rules, reserve_soc=0.0),
    "no grid charging": replace(base_rules, grid_charge_tariff_threshold=0.0),
    "no peak shaving": replace(base_rules, peak_tariff_threshold=10.0),
    "load following": replace(base_rules, generator_mode=GeneratorMode.LOAD_FOLLOWING),
}

registry = StrategyRegistry.default()
for key, rules in VARIANTS.items():
    if key != "optimizer":
        registry.register(key, lambda _cfg, k=key, r=rules: NamedOptimizer(k, r))
ablation_service = SimulationService(registry)

ablation_rows = []
for seed in range(max(4, MC_REPLICATES // 3)):
    rep = ablation_service.run(
        with_param(scenario, "profile", "seed", seed), strategies=list(VARIANTS)
    ).report
    for key in VARIANTS:
        m = rep.metrics[key]
        ablation_rows.append(
            {
                "variant": key,
                "seed": seed,
                "saving_pct": rep.comparison(key).cost_savings_pct,
                "fuel_l": m.fuel_used_l,
                "generator_starts": m.generator_starts,
                "unmet_kwh": m.unmet_energy_kwh,
            }
        )
ablation = pd.DataFrame(ablation_rows)
summary = ablation.groupby("variant")[
    ["saving_pct", "fuel_l", "generator_starts", "unmet_kwh"]
].mean()
summary["Δ saving vs full (pp)"] = summary.saving_pct - summary.loc["optimizer", "saving_pct"]
summary.sort_values("saving_pct", ascending=False)

,saving_pct,fuel_l,generator_starts,unmet_kwh,Δ saving vs full (pp)
variant,,,,,
load following,62.650,10.427,3.300,0.000,0.569
no outage reserve,62.645,23.178,4.550,0.000,0.565
optimizer,62.081,14.609,2.800,0.000,0.000
no grid charging,58.340,34.276,6.650,0.000,-3.741
no peak shaving,58.262,8.148,1.500,0.000,-3.819


In [23]:
fig = px.box(
    ablation,
    x="variant",
    y="saving_pct",
    points="all",
    template="plotly_white",
    title="Ablation: cost saving vs baseline across profiles",
    height=420,
    labels={"saving_pct": "cost saving (%)"},
)
fig.show()

**Answer to RQ4.** Two rules give the optimiser most of its advantage over solar-first:
**charging from the grid at night** and **using the battery when the price is high** (peak
shaving). Without either of them, the saving falls by almost 4 percentage points.

Without the **outage reserve**, the site is a little *cheaper* to run, but it uses more diesel
and starts the generator more often. So the reserve does not save money. Instead, it costs a
little extra to keep backup energy ready. It is a price paid for reliability.

The most surprising result is that **load following** is better than cycle charging for both
cost and fuel at this site. The battery is large enough that the generator runs very rarely, so
running it at an efficient level brings little benefit. This result suggests that the default
rule should be changed. It also shows why each rule should be tested, and not just assumed to
help.

## 10. Many sites (RQ5)

Operators often test a new technology on a few sites and then expect the same result everywhere.
To test this idea, we create a group of different sites (a **fleet**) based on the reference
site. Each site gets its own demand level, power-cut pattern and amount of sunshine. Then we ask:
do the results from the first $n$ sites agree with the results from the whole fleet?

In the table, each row is a fleet made of the first `sites` sites:

* `win_rate` is the share of sites where the optimiser is cheaper than the baseline;
* `ci_low_pct` and `ci_high_pct` are the limits of the 95% bootstrap confidence interval of the
  average saving (there is no interval for a single site);
* `agrees_with_reference` shows whether the fleet's average is inside the confidence interval of
  the largest fleet.

The last cell uses ordinary least squares (OLS) regression. It explains each site's saving with
two values: grid availability and average demand. Both values are **standardised**, which means
they are rescaled to an average of 0 and an SD of 1. So each coefficient means: "how many
percentage points the saving changes when this value increases by one SD". The intercept is the
average saving of the fleet. R² is the share of the differences between sites that the two values
explain.

In [24]:
fleet_template = with_param(scenario, "profile", "days", FLEET_DAYS)
fleet_cfg = FleetConfig.with_sites(FLEET_SITES)
t0 = time.perf_counter()
study = run_fleet_study(fleet_template, fleet_cfg)
print(f"Simulated {len(study.fleet)} sites x 3 strategies in {time.perf_counter() - t0:.1f} s")
display(Markdown(f"> {study.report.verdict()}"))
study.report.to_frame()

Simulated 50 sites x 3 strategies in 3.7 s


> With 50 sites, optimizer saves 59.2% on average versus baseline (95% CI 56.5% to 61.7%) and wins at 100% of sites. Direction is consistent: every fleet size recommends optimizer. Magnitude differs for fleets of 1, 5 and 10 sites: small samples misstate the size of the saving. Estimates are stable from 25 sites upward.

,mean_saving_pct,std_saving_pct,ci_low_pct,ci_high_pct,min_saving_pct,max_saving_pct,mean_fuel_saved_l,win_rate,recommended,agrees_with_reference
sites,,,,,,,,,,
1,71.050,0.000,NaN,NaN,71.050,71.050,183.971,1.000,optimizer,False
5,64.447,6.977,58.969,69.925,57.996,72.622,214.847,1.000,optimizer,False
10,62.587,6.535,58.876,66.602,55.270,72.622,188.063,1.000,optimizer,False
25,58.486,9.039,54.871,61.803,34.966,72.622,185.314,1.000,optimizer,True
50,59.215,9.275,56.548,61.679,34.966,75.461,189.081,1.000,optimizer,True


In [25]:
charts.fleet_convergence(study.report).show()
charts.fleet_distribution(study.fleet).show()
charts.fleet_drivers(study.fleet).show()

In [26]:
# Which site characteristics explain the saving? Ordinary least squares on standardised drivers.
sites = pd.DataFrame(
    {
        "saving_pct": study.fleet.savings_pct("optimizer"),
        "grid_availability": [o.grid_availability for o in study.fleet.outcomes],
        "average_load_kw": [o.average_load_kw for o in study.fleet.outcomes],
    }
)
drivers = ["grid_availability", "average_load_kw"]
X = (sites[drivers] - sites[drivers].mean()) / sites[drivers].std(ddof=0)
X = np.column_stack([np.ones(len(X)), X.to_numpy()])
y = sites.saving_pct.to_numpy()
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
r2 = 1 - ((y - X @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
pd.DataFrame({"standardised effect (pp per SD)": beta}, index=["intercept", *drivers]).assign(
    R2=[r2, np.nan, np.nan]
)

,standardised effect (pp per SD),R2
intercept,59.215,0.682
grid_availability,-2.588,NaN
average_load_kw,-6.911,NaN


**Answer to RQ5.** The verdict above checks two things separately:

* **direction**: does every fleet size choose the same best strategy?
* **size**: is the average saving of every fleet inside the confidence interval of the largest
  fleet?

Every fleet size chooses the optimiser. So the *direction* of the result is correct even with one
site. The *size* of the saving is different: small fleets show a saving that is too high. The
average becomes stable only from about 25 sites.

The regression explains most of the differences between sites. **Demand** has the biggest
effect: at a site with high demand, the same solar panels and battery cover a smaller share of
the energy. Better grid availability also makes the saving smaller, because there is less diesel
to save. So test sites should be chosen to *represent* the whole network in demand and power
cuts, not only because they are easy to reach.

## 11. Discussion

### Limitations of this study

* **Generated data.** The demand, sunshine and power cuts come from mathematical models. We
  checked them against theory (§4.1), but not against measurements from real sites. The package
  can already read real data with `CsvProfileRepository`, so this is the natural next step.
* **Simple rules, no forecasts.** Each strategy looks only at the current hour. None of them uses
  forecasts of weather or demand. A model-predictive controller (MPC), or a mathematical
  optimisation over the whole period, could find the best possible saving. We could then measure
  how far our optimiser is from this best result (its *optimality gap*).
* **Operating costs only.** We do not include the cost of buying (capex), replacing or financing
  equipment. So §7.2 is only a first check. It is not a full life-time cost study (a *net present
  cost* study, which converts all future costs into today's money, as HOMER does).
* **Simple physical models.** The battery has fixed efficiencies and a simple wear cost.
  Temperature, charging speed and ageing over time are not included. The generator uses a
  straight-line fuel formula.
* **One-hour steps.** The model cannot show events that are shorter than one hour, such as the
  start-up of the generator.

### Conclusions

1. At the reference site, the optimiser greatly reduces cost, diesel use and emissions compared
   with the traditional method, and the power supply stays fully reliable (RQ1).
2. The saving depends mainly on how much solar there is compared with the demand. More solar
   stops helping unless the battery also grows, but a larger battery keeps reducing diesel use
   (RQ2).
3. The result stays true with different random weather and power cuts. The optimiser is clearly
   better than the simple solar-first strategy (RQ3).
4. Grid charging at night and battery use at high prices give the optimiser its advantage. The
   outage reserve costs a little extra in exchange for reliability. Load following should replace
   cycle charging as the default for sites with a large battery (RQ4).
5. Results about *which strategy is best* are reliable even from a few sites. Results about *how
   much is saved* need a group of sites that represents the whole network (RQ5).

### References

* Lambert, T., Gilman, P. & Lilienthal, P. (2006). *Micropower system modeling with HOMER*. In
  Farret & Simões, *Integration of Alternative Sources of Energy*, Wiley.
* Barley, C. D. & Winn, C. B. (1996). Optimal dispatch strategy in remote hybrid power systems.
  *Solar Energy*, 58(4–6), 165–179.
* Efron, B. & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
* Gamma, E., Helm, R., Johnson, R. & Vlissides, J. (1994). *Design Patterns*. Addison-Wesley.

## Appendix: saving the results

The scenario, the KPI tables and the experiment data are saved in `results/notebook/`. You can use
these files to recreate the charts or to study the results further.

In [27]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
scenario.save(EXPORT_DIR / "scenario.json")
kpis.to_csv(EXPORT_DIR / "kpis.csv")
results["optimizer"].frame.to_csv(EXPORT_DIR / "optimizer_hourly.csv")
tornado.to_csv(EXPORT_DIR / "tornado.csv")
sweep.to_csv(EXPORT_DIR / "sizing_sweep.csv", index=False)
mc.to_csv(EXPORT_DIR / "monte_carlo.csv")
ablation.to_csv(EXPORT_DIR / "ablation.csv", index=False)
study.fleet.to_frame().to_csv(EXPORT_DIR / "fleet_sites.csv")
sorted(p.name for p in EXPORT_DIR.iterdir())

['ablation.csv',
 'fleet_sites.csv',
 'kpis.csv',
 'monte_carlo.csv',
 'optimizer_hourly.csv',
 'scenario.json',
 'sizing_sweep.csv',
 'tornado.csv']